# Audit static Autodata 3.45 — Drive → Colab → Drive

**Scop:** verificare reproductibilă a celor 16 ZIP-uri din folderul Drive, comparare cu `INVENTAR.json`, clasificare statică a fișierelor `.IDF`, XML, grafice și documentație. Nu este un installer, nu este un emulator și nu execută nimic din arhivă.

**Utilizare:** Runtime → Run all; aprobă autentificarea Google. Notebookul citește doar fișiere accesibile fără ocolirea blocărilor de securitate. Produce exclusiv rapoarte în `Autodata_3_45_ANALYSIS` (în folderul sursă sau în My Drive dacă nu există permisiuni de scriere).

**Limitări:** SHA-256 al RAR-ului original din manifest nu poate fi recalculat din ZIP-urile derivate. Extensia `.IDF` nu implică o schemă cunoscută; fișierele rămân `unknown_undecoded` până la probe verificabile. Nu publicați conținutul proprietar sau fișiere executabile în GitHub.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
from google.auth import default
from pathlib import Path
from collections import Counter
import csv, io, json, os, re, hashlib, zipfile, math, tempfile, traceback, time
creds, _ = default()
drive = build("drive", "v3", credentials=creds, cache_discovery=False)
SOURCE_FOLDER_ID = "1XqpDaYI5zkdUXY85wLSvpsklatT0P0Bd"
WORK = Path("/content/autodata_static_audit")
WORK.mkdir(parents=True, exist_ok=True)
MAX_ENTRY_BYTES = 1024 * 1024 * 1024
MAX_SMALL_BYTES = 12 * 1024 * 1024
SAMPLE_BYTES = 128 * 1024
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
def save_csv(name, rows):
    if not rows: rows=[{"status":"no_rows"}]
    cols=list(dict.fromkeys(k for r in rows for k in r))
    with open(WORK/name,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=cols,extrasaction="ignore");w.writeheader();w.writerows(rows)
def save_json(name,obj):
    (WORK/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
def list_children(parent):
    found=[];token=None
    while True:
        r=drive.files().list(q=f"'{parent}' in parents and trashed = false",fields="nextPageToken,files(id,name,mimeType,size,md5Checksum,capabilities/canDownload)",pageSize=1000,pageToken=token,supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        found.extend(r.get("files",[]));token=r.get("nextPageToken")
        if not token: return found
items=list_children(SOURCE_FOLDER_ID)
save_json("drive_folder_listing.json",items)
save_csv("drive_folder_listing.csv",items)
zips=sorted([x for x in items if re.fullmatch(r"autodata_part_\d{4}\.zip",x["name"])],key=lambda x:x["name"])
print("Folder items:",len(items),"ZIP files:",len(zips))
print("ZIP numbers:",[x["name"] for x in zips])
expected_zip_names=[f"autodata_part_{i:04d}.zip" for i in range(1,17)]
missing_zip_names=sorted(set(expected_zip_names)-{x["name"] for x in zips})
print("Missing expected ZIPs:",missing_zip_names)
# Continue with partial evidence; a missing ZIP cannot be called verified.

In [ ]:
failures=[];downloads=[];zip_paths={}
# Provider's 403 malware/spam block was recorded for part 0016 on 2026-10-30.
# Never acknowledge abuse or attempt a bypass.
KNOWN_BLOCKED={"autodata_part_0016.zip":"Google Drive 403 cannotDownloadAbusiveFile (recorded in source access report)"}
for name in missing_zip_names:
    failures.append({"name":name,"status":"missing","reason":"Not present in Drive listing"})
for name in KNOWN_BLOCKED:
    if name in {x["name"] for x in zips}:
        failures.append({"name":name,"status":"provider_block","reason":KNOWN_BLOCKED[name]})
def download(item):
    name=item["name"]
    if item.get("capabilities",{}).get("canDownload") is False:
        raise PermissionError("Drive canDownload=false")
    target=WORK/name
    req=drive.files().get_media(fileId=item["id"])
    try:
        with open(target,"wb") as output:
            downloader=MediaIoBaseDownload(output,req,chunksize=4*1024*1024)
            done=False
            while not done:
                _,done=downloader.next_chunk()
        return target
    except Exception:
        target.unlink(missing_ok=True)
        raise
for item in [x for x in items if x["name"]=="INVENTAR.json"] + zips:
    if item["name"] in KNOWN_BLOCKED:
        print("SKIPPING provider-blocked:",item["name"])
        continue
    try:
        p=download(item)
        actual_size=p.stat().st_size
        if item.get("size") and actual_size!=int(item["size"]):
            raise ValueError("Downloaded bytes do not match Drive size metadata")
        downloads.append({"name":item["name"],"bytes":actual_size,"sha256":sha256_file(p),"status":"downloaded"})
        if item["name"].endswith(".zip"):zip_paths[item["name"]]=p
    except Exception as exc:
        (WORK/item["name"]).unlink(missing_ok=True)
        failures.append({"name":item["name"],"reason":str(exc)[:500],"status":"blocked_or_unavailable"})
        print("UNAVAILABLE:",item["name"],str(exc)[:160])
save_csv("download_hashes.csv",downloads)
save_json("download_failures.json",failures)
manifest_path=WORK/"INVENTAR.json"
manifest=json.loads(manifest_path.read_text(encoding="utf-8-sig")) if manifest_path.exists() else None
print("Downloaded ZIPs:",len(zip_paths),"Manifest:",bool(manifest))
# No acknowledgeAbuse and no executing any downloaded content.

In [ ]:
zip_entries=[];zip_index={}
for name,p in sorted(zip_paths.items()):
    try:
        with zipfile.ZipFile(p) as zf:
            for info in zf.infolist():
                row={"package":name,"entry":info.filename,"size":info.file_size,"compressed_size":info.compress_size,"crc32":f"{info.CRC:08x}","status":"pending"}
                if info.is_dir():row["status"]="directory"
                elif info.file_size>MAX_ENTRY_BYTES:row["status"]="size_limit"
                elif info.flag_bits & 1:row["status"]="encrypted_skipped"
                else:zip_index[(name,info.filename)]=(p,info.filename)
                zip_entries.append(row)
    except Exception as exc:
        failures.append({"name":name,"reason":"ZIP open: "+str(exc)[:300]})
save_csv("zip_entries.csv",zip_entries)
def chunk_ref(ch):
    return ch.get("package"),ch.get("entry")
def read_entry_stream(pkg,entry,handler,cap=MAX_ENTRY_BYTES):
    p,_=zip_index[(pkg,entry)]
    with zipfile.ZipFile(p) as zf, zf.open(entry) as stream:
        n=0
        while True:
            block=stream.read(1024*1024)
            if not block:break
            n+=len(block)
            if n>cap:raise ValueError("size_limit")
            handler(block)
    return n
print("ZIP entries:",len(zip_entries),"eligible:",len(zip_index))

In [ ]:
manifest_rows=[];static_rows=[];xml_rows=[];graphics_rows=[];docs_rows=[];unknown_rows=[]
def safe_type(path):
    p=path.lower()
    if re.search(r"(^|/)(keygen|activat|crack|patch|emulator)",p) or p.endswith((".exe",".dll",".bat",".cmd",".com",".scr",".msi",".vbs",".ps1",".reg",".lnk")):return "dangerous_hash_only"
    if p.endswith(".idf"):return "idf"
    if p.endswith(".xml"):return "xml"
    if p.endswith((".bmp",".png",".jpg",".jpeg",".gif",".tif",".tiff")):return "graphic"
    if p.endswith((".txt",".rtf",".chm",".pdf",".htm",".html")):return "document"
    return "other"
def entropy(data):
    if not data:return 0
    c=Counter(data);n=len(data)
    return round(-sum((v/n)*math.log2(v/n) for v in c.values()),3)
def sample_strings(b):
    a=re.findall(rb"[\x20-\x7e]{6,}",b)[:12]
    return [s.decode("ascii",errors="replace")[:100] for s in a]
def classify_bytes(path,head,sample,small):
    kind=safe_type(path)
    if kind=="dangerous_hash_only":return
    rec={"path":path,"category":kind,"header64_hex":head[:64].hex(),"sample_entropy":entropy(sample),"sample_ascii_strings":json.dumps(sample_strings(sample),ensure_ascii=False)}
    if kind=="idf":
        rec["format_status"]="unknown_undecoded"
        static_rows.append(rec);unknown_rows.append({"path":path,"reason":"IDF structure not established"})
    elif kind=="xml":
        if small is None:xml_rows.append({"path":path,"status":"too_large_for_xml_parse"});return
        try:
            from defusedxml import ElementTree as DET
            root=DET.fromstring(small)
            tags=Counter(x.tag for x in root.iter())
            xml_rows.append({"path":path,"status":"parsed","root_tag":root.tag,"element_count":sum(tags.values()),"distinct_tags":len(tags),"top_tags":json.dumps(tags.most_common(15),ensure_ascii=False)})
        except Exception as exc:xml_rows.append({"path":path,"status":"parse_failed","reason":str(exc)[:200]})
    elif kind=="graphic":
        if small is None:graphics_rows.append({"path":path,"status":"too_large_for_metadata"});return
        try:
            from PIL import Image
            with Image.open(io.BytesIO(small)) as im:
                graphics_rows.append({"path":path,"status":"metadata_only","format":im.format,"width":im.width,"height":im.height,"mode":im.mode})
        except Exception as exc:graphics_rows.append({"path":path,"status":"metadata_failed","reason":str(exc)[:200]})
    elif kind=="document":docs_rows.append({"path":path,"status":"metadata_and_sample_only","size":len(small) if small is not None else "large"})
    else:
        rec["format_status"]="unclassified"
        static_rows.append(rec);unknown_rows.append({"path":path,"reason":"No supported parser"})
try:
    import defusedxml
except ImportError:
    failures.append({"name":"defusedxml","reason":"Optional XML parser not installed; XML inspection will be reported unavailable","status":"optional_dependency_missing"})
if manifest:
    for i,f in enumerate(manifest.get("files",[]),1):
        path=f.get("path","");chunks=f.get("chunks",[]);h=hashlib.sha256();total=0;problems=[];sample=bytearray();head=b"";small=bytearray();keep_small=(safe_type(path) in ("xml","graphic") and int(f.get("size_bytes",0))<=MAX_SMALL_BYTES)
        for ch in chunks:
            pkg,entry=chunk_ref(ch);chh=hashlib.sha256();size=0
            if (pkg,entry) not in zip_index:
                problems.append(f"missing:{pkg}/{entry}");continue
            def absorb(block):
                h.update(block);chh.update(block)
                if len(sample)<SAMPLE_BYTES:sample.extend(block[:SAMPLE_BYTES-len(sample)])
                if keep_small:small.extend(block)
            try:
                n=read_entry_stream(pkg,entry,absorb)
                total+=n
                if n!=int(ch.get("bytes",-1)):problems.append(f"chunk_size_mismatch:{pkg}/{entry}")
                if ch.get("sha256") and chh.hexdigest().lower()!=ch["sha256"].lower():problems.append(f"chunk_hash_mismatch:{pkg}/{entry}")
            except Exception as exc:problems.append(f"read_failed:{pkg}/{entry}:{str(exc)[:80]}")
        if total!=int(f.get("size_bytes",-1)):problems.append("file_size_mismatch")
        status="PASS" if not problems else "FAIL_OR_UNAVAILABLE"
        manifest_rows.append({"path":path,"expected_size":f.get("size_bytes"),"actual_size":total,"sha256_reconstructed":h.hexdigest() if not problems else "","chunk_count":len(chunks),"status":status,"problems":"; ".join(problems),"category":safe_type(path)})
        if status=="PASS" and safe_type(path)!="dangerous_hash_only":
            classify_bytes(path,bytes(sample[:64]),bytes(sample),bytes(small) if keep_small else None)
        if i%20==0:print("Processed manifest files:",i)
else:
    failures.append({"name":"INVENTAR.json","reason":"Manifest inaccessible; cannot verify reconstructed logical files"})
save_csv("manifest_vs_actual.csv",manifest_rows)
save_csv("static_analysis.csv",static_rows)
save_csv("unknown_formats.csv",unknown_rows)
save_csv("xml_summary.csv",xml_rows)
save_csv("graphics_summary.csv",graphics_rows)
save_csv("documentation_summary.csv",docs_rows)
save_json("download_failures.json",failures)
print("Manifest records:",len(manifest_rows),"PASS:",sum(x["status"]=="PASS" for x in manifest_rows),"FAIL:",sum(x["status"]!="PASS" for x in manifest_rows))

In [ ]:
summary={"source_folder":SOURCE_FOLDER_ID,"zip_count_in_folder":len(zips),"zip_downloaded":len(zip_paths),"manifest_file_count":len(manifest.get("files",[])) if manifest else None,"files_checked":len(manifest_rows),"files_pass":sum(r["status"]=="PASS" for r in manifest_rows),"files_fail_or_unavailable":sum(r["status"]!="PASS" for r in manifest_rows),"idf_unknown":sum(r["category"]=="idf" for r in static_rows),"download_failures":len(failures),"note":"RAR original SHA256 not rechecked; no executables were run; IDF not decoded"}
save_json("summary.json",summary)
report="# Autodata 3.45 static audit\n\n"+ "\n".join(f"- {k}: {v}" for k,v in summary.items())+"\n\nInterpretation: PASS means reconstructed bytes and manifest chunk sizes/hashes matched where provided; unknown IDF format is not decoded.\n"
(WORK/"REPORT.md").write_text(report,encoding="utf-8")
# Upload only generated reports; never upload or re-share ZIPs, reconstructed binaries or raw samples.
def create_report_folder():
    existing=[x for x in list_children(SOURCE_FOLDER_ID) if x["name"]=="Autodata_3_45_ANALYSIS" and x["mimeType"]=="application/vnd.google-apps.folder"]
    if existing:return existing[0]["id"]
    body={"name":"Autodata_3_45_ANALYSIS","mimeType":"application/vnd.google-apps.folder","parents":[SOURCE_FOLDER_ID]}
    try:return drive.files().create(body=body,fields="id").execute()["id"]
    except Exception:
        return drive.files().create(body={"name":"Autodata_3_45_ANALYSIS","mimeType":"application/vnd.google-apps.folder"},fields="id").execute()["id"]
report_folder=create_report_folder()
allowed={"drive_folder_listing.json","drive_folder_listing.csv","download_hashes.csv","zip_entries.csv","manifest_vs_actual.csv","static_analysis.csv","unknown_formats.csv","xml_summary.csv","graphics_summary.csv","documentation_summary.csv","download_failures.json","summary.json","REPORT.md"}
for name in sorted(allowed):
    p=WORK/name
    if not p.exists():continue
    media=MediaIoBaseUpload(io.BytesIO(p.read_bytes()),mimetype="text/plain",resumable=True)
    drive.files().create(body={"name":name,"parents":[report_folder]},media_body=media,fields="id").execute()
print("AUDIT REPORT FOLDER: https://drive.google.com/drive/folders/"+report_folder)
print(json.dumps(summary,ensure_ascii=False,indent=2))